# Baseline trajectory inference

Here, we use Palantir and CellRank as baseline methods for trajectory inference (specifically: cell fate inference) against which we can compare the bowside fate probabilities.
We use the synthetic curated datasets from BEELINE ([Pratapa et al. (2020)](https://doi.org/10.1038/s41592-019-0690-6)), downloaded from [Zenodo](https://doi.org/10.5281/zenodo.19009603) with the same expression matrices as used for bowside.

In [1]:
from pathlib import Path

import cellrank as cr
import numpy as np
import pandas as pd
import palantir
import scanpy as sc
from cellrank.estimators import GPCCA
from cellrank.kernels import ConnectivityKernel, PseudotimeKernel

In [2]:
# Load AnnData objects
dataset_names = ["GSD", "HSC", "mCAD", "VSC"]
adatas = {
    name: [
        sc.read_h5ad(
            f"../data/BEELINE-data/processed/Curated/{name}/{name}_processed_{i + 1}.h5ad"
        )
        for i in range(10)
    ]
    for name in dataset_names
}
for key, lst in adatas.items():
    print(key)
    display(lst[0])

GSD


AnnData object with n_obs × n_vars = 2000 × 19
    obs: 'PseudoTime1', 'PseudoTime2', 'pseudotime'
    var: 'mean', 'std'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap'
    layers: 'log1p'
    obsp: 'connectivities', 'distances'

HSC


AnnData object with n_obs × n_vars = 2000 × 11
    obs: 'PseudoTime1', 'PseudoTime2', 'PseudoTime3', 'PseudoTime4', 'pseudotime'
    var: 'mean', 'std'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap'
    layers: 'log1p'
    obsp: 'connectivities', 'distances'

mCAD


AnnData object with n_obs × n_vars = 2000 × 5
    obs: 'PseudoTime1', 'PseudoTime2', 'pseudotime'
    var: 'mean', 'std'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap'
    layers: 'log1p'
    obsp: 'connectivities', 'distances'

VSC


AnnData object with n_obs × n_vars = 2000 × 8
    obs: 'PseudoTime1', 'PseudoTime2', 'PseudoTime3', 'PseudoTime4', 'PseudoTime5', 'pseudotime'
    var: 'mean', 'std'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap'
    layers: 'log1p'
    obsp: 'connectivities', 'distances'

In [3]:
# Create output directories
out_dirs_palantir = {
    name: Path(f"../output/BEELINE_{name}/palantir") for name in dataset_names
}
for out_dir in out_dirs_palantir.values():
    out_dir.mkdir(parents=True, exist_ok=True)
out_dirs_cellrank = {
    name: Path(f"../output/BEELINE_{name}/cellrank") for name in dataset_names
}
for out_dir in out_dirs_cellrank.values():
    out_dir.mkdir(parents=True, exist_ok=True)

In [4]:
# Run Palantir on all datasets, using the log1p expression instead of a PCA
for name, lst in adatas.items():
    for i, adata in enumerate(lst):
        adata.obsm["X_log1p"] = adata.layers["log1p"]
        palantir.utils.run_diffusion_maps(
            adata, pca_key="X_log1p", n_components=5, knn=30
        )
        palantir.utils.determine_multiscale_space(adata)
        early_cell = adata.obs_names[adata.obs["pseudotime"].argmin()]  # type: ignore
        # Determine terminal cells from pseudotime columns (fixes number of trajectories!)
        pt_cols = [c for c in adata.obs.columns if c.startswith("PseudoTime")]
        terminal_states = pd.Series({adata.obs[c].idxmax(): c for c in pt_cols})

        palantir.core.run_palantir(
            adata,
            early_cell=early_cell,
            terminal_states=terminal_states,
            knn=30,
            num_waypoints=500,
            save_as_df=False,
        )

        # Save pseudotime, fate probabilities, and entropy
        palantir_res = sc.get.obs_df(
            adata,
            keys=["palantir_pseudotime", "palantir_entropy"],
            obsm_keys=[
                ("palantir_fate_probabilities", col)
                for col in range(adata.obsm["palantir_fate_probabilities"].shape[1])
            ],
        )
        palantir_res.rename(
            columns={
                f"palantir_fate_probabilities-{col}": f"fate_probability_{col}"
                for col in range(adata.obsm["palantir_fate_probabilities"].shape[1])
            },
            inplace=True,
        )
        palantir_res.to_parquet(
            out_dirs_palantir[name] / f"fate_probabilities_{i}.parquet"
        )

Sampling and flocking waypoints...
Time for determining waypoints: 0.0004824360211690267 minutes
Determining pseudotime...
Shortest path distances using 30-nearest neighbor graph...
Time for shortest paths: 0.0738100290298462 minutes
Iteratively refining the pseudotime...
Correlation at iteration 1: 1.0000
Entropy and branch probabilities...
Markov chain construction...
Computing fundamental matrix and absorption probabilities...
Project results to all cells...
Sampling and flocking waypoints...
Time for determining waypoints: 0.0004755814870198568 minutes
Determining pseudotime...
Shortest path distances using 30-nearest neighbor graph...
Time for shortest paths: 0.006804947058359782 minutes
Iteratively refining the pseudotime...
Correlation at iteration 1: 1.0000
Entropy and branch probabilities...
Markov chain construction...
Computing fundamental matrix and absorption probabilities...
Project results to all cells...
Sampling and flocking waypoints...
Time for determining waypoints:

/Users/janschleicher/miniconda3/envs/palantir/lib/python3.12/site-packages/palantir/core.py:709: UserWarning: Some of the cells were unreachable. Consider increasing the k for 
             nearest neighbor graph construction.
  warnings.warn(


Time for shortest paths: 0.009401750564575196 minutes
Iteratively refining the pseudotime...
Correlation at iteration 1: 1.0000
Entropy and branch probabilities...
Markov chain construction...
Computing fundamental matrix and absorption probabilities...
Project results to all cells...


In [5]:
# Run CellRank on all datasets
for name, lst in adatas.items():
    for i, adata in enumerate(lst):
        # Use the pseudotime kernel
        pk = PseudotimeKernel(adata, time_key="pseudotime")
        pk.compute_transition_matrix()
        g_pk = GPCCA(pk)
        g_pk.fit(n_states=[3, 20])
        g_pk.predict_terminal_states()
        g_pk.compute_fate_probabilities()  # type: ignore
        fate_probabilities_arr = np.asarray(adata.obsm["lineages_fwd"])
        fate_probabilities = pd.DataFrame(
            fate_probabilities_arr,
            index=adata.obs_names,
            columns=[
                f"fate_probability_{i}" for i in range(fate_probabilities_arr.shape[1])
            ],
        )
        fate_probabilities.to_parquet(
            out_dirs_cellrank[name] / f"fate_probabilities_pseudotime_{i}.parquet"
        )

        # Use the connectivity kernel
        ck = ConnectivityKernel(adata)
        ck.compute_transition_matrix()
        g_ck = GPCCA(ck)
        g_ck.fit(n_states=[3, 20])
        g_ck.predict_terminal_states()
        g_ck.compute_fate_probabilities()  # type: ignore
        fate_probabilities_arr = np.asarray(adata.obsm["lineages_fwd"])
        fate_probabilities = pd.DataFrame(
            fate_probabilities_arr,
            index=adata.obs_names,
            columns=[
                f"fate_probability_{i}" for i in range(fate_probabilities_arr.shape[1])
            ],
        )
        fate_probabilities.to_parquet(
            out_dirs_cellrank[name] / f"fate_probabilities_connectivity_{i}.parquet"
        )

  0%|          | 0/2000 [00:00<?, ?cell/s]

  0%|          | 0/3 [00:00<?, ?/s]

  0%|          | 0/3 [00:00<?, ?/s]

  0%|          | 0/2000 [00:00<?, ?cell/s]

  0%|          | 0/3 [00:00<?, ?/s]

...

  0%|          | 0/3 [00:00<?, ?/s]

  0%|          | 0/3 [00:00<?, ?/s]

  0%|          | 0/2000 [00:00<?, ?cell/s]

  0%|          | 0/3 [00:00<?, ?/s]

  0%|          | 0/3 [00:00<?, ?/s]